# Library API Manual Check

이 노트북은 현재 `modeling_module` public API를 직접 눈으로 확인하면서 테스트하기 위한 수동 점검용 notebook입니다.

## 가정
- `tb_master_target`: endogenous-only 테스트용 원천 테이블
- `tb_master_exo`: endogenous + exogenous 테스트용 원천 테이블
- 현재 라이브러리는 **one-table exogenous** 경로를 지원하므로, `tb_master_exo`에 `y`가 없으면 `tb_master_target`의 `y`를 join해서 하나의 long table로 맞춥니다.

## 이 노트북에서 확인하는 것
1. endogenous-only dataloader/build_dataset/train/load_predictor smoke
2. endogenous + exogenous one-table dataloader/train/load_predictor smoke
3. batch shape, checkpoint path, prediction 결과 길이 확인

## 먼저 할 일
- remote server kernel이면 첫 번째 code cell의 `REPO_ROOT_OVERRIDE`를 서버 기준 절대경로로 먼저 설정합니다.
- 아래 `TARGET_SOURCE`, `EXO_SOURCE`, 컬럼명, exogenous 컬럼 목록을 실제 데이터에 맞게 수정합니다.
- 로컬 parquet/csv가 아니라 DB 테이블을 직접 읽는 환경이면, 로딩 셀만 DB connector 코드로 바꿔서 `polars.DataFrame`을 반환하면 됩니다.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch


# Remote kernel이면 여기에 서버 기준 repo 절대경로를 넣을 수 있습니다.
# Example: REPO_ROOT_OVERRIDE = Path("/home/ubuntu/ts_forecaster_lib")
# repo clone이 서버에 없고 modeling_module만 설치되어 있어도 notebook는 동작하도록 구성합니다.
REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src").exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = "ts_forecaster_lib", max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return

    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root

    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get("TS_FORECASTER_REPO_ROOT")

    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])

    home = Path.home()
    common_roots = [
        home,
        home / "workspace",
        home / "workspaces",
        home / "projects",
        home / "PycharmProjects",
        Path("/workspace"),
        Path("/workspaces"),
        Path("/home"),
        Path("/root"),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate

    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / "src" if repo_root is not None else None

    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    spec = importlib.util.find_spec("modeling_module")
    if spec is None or spec.origin is None:
        raise RuntimeError(
            "Could not import modeling_module. Either set REPO_ROOT_OVERRIDE to the server repo path, "
            "set TS_FORECASTER_REPO_ROOT, or install the package on the remote environment."
        )

    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent

    if repo_root is None and module_root.parent.name == "src":
        repo_root = module_root.parent.parent
        src_root = repo_root / "src"

    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from modeling_module import (
    ArtifactConfig,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,
    RuntimeConfig,
    SSLConfig,
    TrainRequest,
    TrainerConfig,
    build_dataloader,
    build_dataset,
    load_predictor,
    train,
)
from modeling_module.utils.device import select_default_device


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEFAULT_DEVICE, DEFAULT_DEVICE_DIAGNOSTIC = select_default_device()

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)
print("PYTHON   :", sys.executable)
print("TORCH    :", torch.__version__)
print("DEVICE   :", DEFAULT_DEVICE)
if DEFAULT_DEVICE_DIAGNOSTIC:
    print("DEVICE_NOTE:", DEFAULT_DEVICE_DIAGNOSTIC)


## 1. Source / Column Config

기본값은 현재 repo의 `tb_master_target` / `tb_master_exo` 스키마에 맞춰 두었습니다.
remote/DB 환경이면 아래 셀에서 경로와 컬럼만 바꿔서 그대로 재사용하면 됩니다.

### 로딩 방식
- 로컬 파일 기준: `TARGET_SOURCE`, `EXO_SOURCE` 에 parquet/csv 경로를 넣습니다.
- DB 기준: 아래 helper를 쓰지 말고, 로딩 셀에서 직접 query 후 `polars.DataFrame`을 반환하면 됩니다.

### 컬럼 의미
- `ID_COL`: 시계열 식별자
- `DATE_COL`: 시점 컬럼
- `Y_COL`: target
- `PAST_EXO_CONT_COLS`: lookback 구간에서 모델에 들어갈 과거 연속형 외생변수
- `FUTURE_EXO_CONT_COLS`: horizon 구간에서 known future covariate로 들어갈 연속형 외생변수
- `PAST_EXO_CAT_COLS`: 필요 시 범주형 과거 외생변수
- 아래 예시는 `DataRequest(...)`, `TrainRequest(...)` dataclass를 직접 만드는 방식입니다.
- 내부 구조는 `window / columns / exogenous / loader` 와 `trainer / ssl / runtime / artifacts` 로 나뉩니다.

### 현재 기본 외생변수 해석
- 과거 연속형: seasonal sin/cos + `weather_index` + `macro_index` + `promo_strength` + `part_len` + `week_of_year`
- 미래 known covariate: seasonal sin/cos + `weather_index` + `macro_index` + `promo_strength` + `week_of_year` + promo/outage/season flag
- `part_prefix_2` 는 현재 스냅샷에서 상수라 기본값에서 제외했습니다.
- `part_group_id` 는 필요하면 `PAST_EXO_CAT_COLS` 에 추가할 수 있습니다.


In [ ]:
# Example:
# if REPO_ROOT is not None:
#     TARGET_SOURCE = REPO_ROOT / "raw_data" / "exports" / "tb_master_target.parquet"
#     EXO_SOURCE = REPO_ROOT / "raw_data" / "exports" / "tb_master_exo.parquet"

DATA_ROOT = REPO_ROOT / "raw_data" / "master" if REPO_ROOT is not None else None
TARGET_SOURCE = DATA_ROOT / "tb_master_target.parquet" if DATA_ROOT is not None else None
EXO_SOURCE = DATA_ROOT / "tb_master_exo.parquet" if DATA_ROOT is not None else None

FREQ = "weekly"
ID_COL = "oper_part_no"
DATE_COL = "demand_dt"
Y_COL = "demand_qty"

# tb_master_exo에 y가 없으면 tb_master_target의 y를 join합니다.
JOIN_TARGET_INTO_EXO = True

PAST_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "part_len",
    "week_of_year",
]
FUTURE_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "week_of_year",
    "promo_flag",
    "supply_outage_flag",
    "peak_season_flag",
    "is_year_start",
    "is_year_end",
    "is_q_start",
    "is_q_end",
]
PAST_EXO_CAT_COLS = []
# Optional categorical candidate: ["part_group_id"]

LOOKBACK = 52
HORIZON = 27
BATCH_SIZE = 16
MAX_IDS = 32
MIN_OBSERVED_TARGET_ROWS = LOOKBACK + HORIZON

TRAIN_EPOCHS = 1
TRAIN_LR = 1e-3
TRAIN_DEVICE = DEFAULT_DEVICE

ENDO_MODELS = ["patchtst_base"]
EXO_MODELS = ["patchtst_base"]
# ExoTST를 쓰고 싶으면 보통 둘 다 필요합니다.
# EXO_MODELS = ["exotst_base"]

ARTIFACT_BASE = REPO_ROOT if REPO_ROOT is not None else NOTEBOOK_DIR
ARTIFACT_ROOT = ARTIFACT_BASE / "artifacts" / "notebook_manual_checks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

config_snapshot = {
    "TARGET_SOURCE": str(TARGET_SOURCE) if TARGET_SOURCE is not None else None,
    "EXO_SOURCE": str(EXO_SOURCE) if EXO_SOURCE is not None else None,
    "FREQ": FREQ,
    "ID_COL": ID_COL,
    "DATE_COL": DATE_COL,
    "Y_COL": Y_COL,
    "PAST_EXO_CONT_COLS": PAST_EXO_CONT_COLS,
    "FUTURE_EXO_CONT_COLS": FUTURE_EXO_CONT_COLS,
    "PAST_EXO_CAT_COLS": PAST_EXO_CAT_COLS,
    "LOOKBACK": LOOKBACK,
    "HORIZON": HORIZON,
    "BATCH_SIZE": BATCH_SIZE,
    "ENDO_MODELS": ENDO_MODELS,
    "EXO_MODELS": EXO_MODELS,
    "TRAIN_DEVICE": TRAIN_DEVICE,
}
print(json.dumps(config_snapshot, indent=2, ensure_ascii=False))


## 2. Helper Functions

이 셀은 실제 테스트를 위한 공통 helper입니다.
- 파일/데이터프레임 로딩
- 컬럼 검증
- history가 충분한 id만 샘플링
- `tb_master_exo`를 one-table training 형태로 변환
- batch shape 출력


In [ ]:
def load_polars_table(source, table_name: str) -> pl.DataFrame:
    if isinstance(source, pl.DataFrame):
        return source.clone()
    if source is None:
        raise ValueError(
            f"{table_name} source is None. Set a parquet/csv path or replace this loader with a DB query."
        )

    path = Path(source)
    if not path.exists():
        raise FileNotFoundError(f"{table_name} source not found: {path}")

    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pl.read_parquet(path)
    if suffix in {".csv", ".txt"}:
        return pl.read_csv(path)

    raise ValueError(f"Unsupported file type for {table_name}: {path}")


def assert_columns(df: pl.DataFrame, required: list[str], table_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def sample_ids_by_observed_target(
    df: pl.DataFrame,
    *,
    id_col: str,
    y_col: str,
    min_obs: int,
    max_ids: int | None,
) -> list[str]:
    counts = (
        df.group_by(id_col)
        .agg(pl.col(y_col).is_not_null().sum().alias("observed_target_rows"))
        .filter(pl.col("observed_target_rows") >= min_obs)
        .sort("observed_target_rows", descending=True)
    )

    if max_ids is not None:
        counts = counts.head(max_ids)

    ids = counts[id_col].cast(pl.String).to_list()
    if not ids:
        raise ValueError(
            "No ids have enough observed target history. "
            f"Need at least {min_obs} non-null `{y_col}` rows per id."
        )
    return ids


def prepare_target_df(target_df: pl.DataFrame) -> pl.DataFrame:
    required = [ID_COL, DATE_COL, Y_COL]
    assert_columns(target_df, required, "tb_master_target")

    df = (
        target_df.select(required)
        .drop_nulls([ID_COL, DATE_COL, Y_COL])
        .with_columns(pl.col(ID_COL).cast(pl.String))
        .sort([ID_COL, DATE_COL])
    )

    sampled_ids = sample_ids_by_observed_target(
        df,
        id_col=ID_COL,
        y_col=Y_COL,
        min_obs=MIN_OBSERVED_TARGET_ROWS,
        max_ids=MAX_IDS,
    )
    return df.filter(pl.col(ID_COL).is_in(sampled_ids))


def prepare_exo_one_table(target_df: pl.DataFrame, exo_df: pl.DataFrame) -> pl.DataFrame:
    exo_base = exo_df.with_columns(pl.col(ID_COL).cast(pl.String)).sort([ID_COL, DATE_COL])

    if Y_COL not in exo_base.columns:
        if not JOIN_TARGET_INTO_EXO:
            raise ValueError(
                "tb_master_exo does not contain y. Set JOIN_TARGET_INTO_EXO=True or join y yourself first."
            )
        exo_base = exo_base.join(
            target_df.select([ID_COL, DATE_COL, Y_COL]),
            on=[ID_COL, DATE_COL],
            how="left",
        )

    required = [
        ID_COL,
        DATE_COL,
        Y_COL,
        *PAST_EXO_CONT_COLS,
        *PAST_EXO_CAT_COLS,
        *FUTURE_EXO_CONT_COLS,
    ]
    required = list(dict.fromkeys(required))
    assert_columns(exo_base, required, "tb_master_exo(one-table)")

    sampled_ids = sample_ids_by_observed_target(
        exo_base,
        id_col=ID_COL,
        y_col=Y_COL,
        min_obs=MIN_OBSERVED_TARGET_ROWS,
        max_ids=MAX_IDS,
    )

    return (
        exo_base.select(required)
        .filter(pl.col(ID_COL).is_in(sampled_ids))
        .sort([ID_COL, DATE_COL])
    )


def describe_batch(batch, name: str) -> None:
    x = batch[0] if len(batch) >= 1 else None
    y = batch[1] if len(batch) >= 2 else None
    part_ids = batch[2] if len(batch) >= 3 else None
    future_exo = batch[3] if len(batch) >= 4 else None
    past_exo_cont = batch[4] if len(batch) >= 5 else None
    past_exo_cat = batch[5] if len(batch) >= 6 else None

    print(f"[{name}] tuple_len      : {len(batch)}")
    if x is not None:
        print(f"[{name}] x shape       : {tuple(x.shape)}")
    if y is not None and torch.is_tensor(y):
        print(f"[{name}] y shape       : {tuple(y.shape)}")
    if future_exo is not None and torch.is_tensor(future_exo):
        print(f"[{name}] future_exo    : {tuple(future_exo.shape)}")
    if past_exo_cont is not None and torch.is_tensor(past_exo_cont):
        print(f"[{name}] past_exo_cont : {tuple(past_exo_cont.shape)}")
    if past_exo_cat is not None and torch.is_tensor(past_exo_cat):
        print(f"[{name}] past_exo_cat  : {tuple(past_exo_cat.shape)}")
    if part_ids is not None:
        preview = part_ids[:3] if isinstance(part_ids, list) else part_ids
        print(f"[{name}] ids preview   : {preview}")


## 3. Endogenous-only: `tb_master_target`

이 섹션은 target만 있는 데이터로 현재 public API를 확인합니다.

체크 포인트:
- `build_dataset()` / `build_dataloader()` 가 바로 되는지
- batch shape가 기대와 맞는지
- `train()` 결과에서 checkpoint가 생성되는지
- `load_predictor()` 로 같은 checkpoint를 다시 불러올 수 있는지


In [ ]:
target_raw = load_polars_table(TARGET_SOURCE, "tb_master_target")
target_df = prepare_target_df(target_raw)

print("target_raw shape:", target_raw.shape)
print("target_df shape :", target_df.shape)
target_df.head(10)


In [ ]:
endo_data_req = DataRequest(
    df=target_df,
    window=DataWindowConfig(
        lookback=LOOKBACK,
        horizon=HORIZON,
        freq=FREQ,
    ),
    columns=DataColumnConfig(
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    ),
    loader=LoaderConfig(
        stage="train",
        batch_size=BATCH_SIZE,
        shuffle=True,
    ),
)

endo_train_dataset = build_dataset(endo_data_req)
endo_train_loader = build_dataloader(endo_data_req)
endo_batch = next(iter(endo_train_loader))

print("endo_train_dataset len:", len(endo_train_dataset))
describe_batch(endo_batch, "ENDO/train")


In [ ]:
endo_save_dir = ARTIFACT_ROOT / "endo_only"
endo_save_dir.mkdir(parents=True, exist_ok=True)

endo_train_req = TrainRequest(
    data=DataRequest(
        df=target_df,
        window=DataWindowConfig(
            lookback=LOOKBACK,
            horizon=HORIZON,
            freq=FREQ,
        ),
        columns=DataColumnConfig(
            id_col=ID_COL,
            date_col=DATE_COL,
            y_col=Y_COL,
        ),
        loader=LoaderConfig(
            batch_size=BATCH_SIZE,
        ),
    ),
    models=ENDO_MODELS,
    trainer=TrainerConfig(
        epochs=TRAIN_EPOCHS,
        lr=TRAIN_LR,
    ),
    runtime=RuntimeConfig(
        device=TRAIN_DEVICE,
    ),
    artifacts=ArtifactConfig(
        save_dir=str(endo_save_dir),
        auto_save_dir=False,
    ),
)

endo_result = train(endo_train_req)

print("requested_models:", endo_result.requested_models)
print("primary_ckpt_path:", endo_result.primary_ckpt_path)
print("manifest_path    :", endo_result.manifest_path)
print(json.dumps(endo_result.ckpt_paths, indent=2, ensure_ascii=False))


In [ ]:
endo_predictor = load_predictor(endo_result.primary_ckpt_path, device=TRAIN_DEVICE)
endo_pred = endo_predictor(endo_batch, horizon=HORIZON)

print("predictor.model_key:", endo_predictor.model_key)
print("prediction keys    :", list(endo_pred.keys()))
print("point length       :", len(endo_pred["point"]))
print("expected length    :", int(endo_batch[0].shape[0]) * HORIZON)
endo_pred


## 4. Endogenous + Exogenous: `tb_master_exo`

이 섹션은 one-table exogenous 경로를 확인합니다.

현재 라이브러리 기준 해석:
- `past_exo_*` 는 lookback 구간에서 slice
- `future_exo_cont_cols` 는 horizon 구간에서 known future covariate로 slice
- `tb_master_exo` 에 `y`가 없으면 `tb_master_target`의 `y`를 join

주의:
- `EXO_MODELS = ["exotst_base"]` 로 돌리려면 보통 `PAST_EXO_CONT_COLS` 와 `FUTURE_EXO_CONT_COLS` 를 둘 다 채우는 것이 안전합니다.
- `patchtst_base`, `patchmixer_base`, `titan_lmm` 은 좀 더 유연합니다.


In [ ]:
if not (PAST_EXO_CONT_COLS or PAST_EXO_CAT_COLS or FUTURE_EXO_CONT_COLS):
    raise ValueError(
        "Set at least one of PAST_EXO_CONT_COLS / PAST_EXO_CAT_COLS / FUTURE_EXO_CONT_COLS before running the exogenous section."
    )

exo_raw = load_polars_table(EXO_SOURCE, "tb_master_exo")
exo_one_table = prepare_exo_one_table(target_df=target_df, exo_df=exo_raw)

print("exo_raw shape      :", exo_raw.shape)
print("exo_one_table shape:", exo_one_table.shape)
print("exo_one_table cols :", exo_one_table.columns)
exo_one_table.head(10)


In [ ]:
exo_data_req = DataRequest(
    df=exo_one_table,
    window=DataWindowConfig(
        lookback=LOOKBACK,
        horizon=HORIZON,
        freq=FREQ,
    ),
    columns=DataColumnConfig(
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    ),
    exogenous=ExogenousConfig(
        use_exogenous_mode=True,
        past_exo_cont_cols=PAST_EXO_CONT_COLS,
        past_exo_cat_cols=PAST_EXO_CAT_COLS,
        future_exo_cont_cols=FUTURE_EXO_CONT_COLS,
    ),
    loader=LoaderConfig(
        stage="train",
        batch_size=BATCH_SIZE,
        shuffle=True,
    ),
)

exo_train_dataset = build_dataset(exo_data_req)
exo_train_loader = build_dataloader(exo_data_req)
exo_batch = next(iter(exo_train_loader))

print("exo_train_dataset len:", len(exo_train_dataset))
describe_batch(exo_batch, "EXO/train")


In [ ]:
exo_save_dir = ARTIFACT_ROOT / "endo_plus_exo"
exo_save_dir.mkdir(parents=True, exist_ok=True)

exo_train_req = TrainRequest(
    data=DataRequest(
        df=exo_one_table,
        window=DataWindowConfig(
            lookback=LOOKBACK,
            horizon=HORIZON,
            freq=FREQ,
        ),
        columns=DataColumnConfig(
            id_col=ID_COL,
            date_col=DATE_COL,
            y_col=Y_COL,
        ),
        exogenous=ExogenousConfig(
            use_exogenous_mode=True,
            past_exo_cont_cols=PAST_EXO_CONT_COLS,
            past_exo_cat_cols=PAST_EXO_CAT_COLS,
            future_exo_cont_cols=FUTURE_EXO_CONT_COLS,
        ),
        loader=LoaderConfig(
            batch_size=BATCH_SIZE,
        ),
    ),
    models=EXO_MODELS,
    trainer=TrainerConfig(
        epochs=TRAIN_EPOCHS,
        lr=TRAIN_LR,
    ),
    # SSL을 같이 보려면 아래 주석을 해제하세요.
    # ssl=SSLConfig(
    #     mode="full",
    #     pretrain_epochs=5,
    # ),
    runtime=RuntimeConfig(
        device=TRAIN_DEVICE,
    ),
    artifacts=ArtifactConfig(
        save_dir=str(exo_save_dir),
        auto_save_dir=False,
    ),
)

exo_result = train(exo_train_req)

print("requested_models:", exo_result.requested_models)
print("primary_ckpt_path:", exo_result.primary_ckpt_path)
print("manifest_path    :", exo_result.manifest_path)
print(json.dumps(exo_result.ckpt_paths, indent=2, ensure_ascii=False))


In [ ]:
exo_predictor = load_predictor(exo_result.primary_ckpt_path, device=TRAIN_DEVICE)
exo_pred = exo_predictor(exo_batch, horizon=HORIZON)

print("predictor.model_key:", exo_predictor.model_key)
print("prediction keys    :", list(exo_pred.keys()))
print("point length       :", len(exo_pred["point"]))
print("expected length    :", int(exo_batch[0].shape[0]) * HORIZON)
exo_pred


## 5. Troubleshooting

자주 보게 되는 실패 원인:
- `lookback` 이 frequency별 `patch_len` 보다 짧음
- `tb_master_exo` 에 `y`가 없는데 join을 하지 않음
- `future_exo_cont_cols` 를 넣었지만 실제 컬럼명이 다름
- `exotst_base` 를 쓰면서 `past_exo_cont_cols` 또는 `future_exo_cont_cols` 가 비어 있음
- 데이터 한 id당 관측 target row 수가 `lookback + horizon` 보다 부족함

추천 순서:
1. 먼저 endogenous-only 섹션으로 batch/train/predict가 도는지 확인
2. 그 다음 exogenous 섹션에서 batch shape만 먼저 확인
3. 마지막으로 exogenous training/predict를 실행
